# 02 · Economic calendar audit

Post-CSV cleaning checks. **PIT disclaimer:** the research CSV is **not**
vintage-safe. Live/paper must not read this CSV — see algorithm notes
**Live Event Calendar Viability**.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.economic_calendar_fetcher import (
    DEFAULT_CSV,
    UNKNOWN_EVENTS_CSV,
    list_event_ids,
    load_economic_calendar,
)
from data.processing.s3_fx_event_panel import build_s3_event_panel
from data.processing.s3_fx_price_panel import s3_data_dir

DATA_DIR = s3_data_dir(ROOT)
print("CSV=", DEFAULT_CSV, "exists=", os.path.isfile(DEFAULT_CSV))


## 1. Load + range / impact mix


In [ ]:
cal = load_economic_calendar(refresh=True)
print("rows=", len(cal))
if not cal.empty:
    ts = pd.to_datetime(cal["date"] if "date" in cal.columns else cal.iloc[:, 0])
    print("date range:", ts.min(), "→", ts.max())
    if "impact" in cal.columns:
        display(cal["impact"].value_counts().sort_index())
    print("n event_ids=", len(list_event_ids(cal)))


## 2. Unknown events + missing-release audit


In [ ]:
if os.path.isfile(UNKNOWN_EVENTS_CSV):
    unk = pd.read_csv(UNKNOWN_EVENTS_CSV)
    print("unknown_events.csv rows=", len(unk))
    display(unk.head(40))
else:
    print("no unknown_events.csv yet (cleaner may create on load)")

# Top-ish G10 releases we expect mapped
expected = [
    "us_nfp", "us_cpi_yoy", "us_fomc_rate", "us_gdp_qoq", "us_ism_mfg",
    "eu_cpi_yoy", "gb_cpi_yoy", "jp_cpi_yoy",
]
have = set(list_event_ids(cal)) if not cal.empty else set()
missing = [e for e in expected if e not in have]
print("missing expected ids (soft audit):", missing)


## 3. sigma_N distribution (N=20)


In [ ]:
events = build_s3_event_panel(cal, N=20, cache=False, data_dir=DATA_DIR)
if not events.empty and "sigma_N" in events.columns:
    display(events["sigma_N"].describe())
    display(events["z"].describe())
else:
    print("empty event panel")


## 4. PIT disclaimer

Research `economic_calendar.csv` actual/forecast/previous are **not** vintage-PIT.
Do not paper-trade the event sleeve off this CSV alone.
